# 01 — Naive RAG vs. Advanced RAG

**Goal**: Understand the basic RAG flow and its limitations.

This notebook builds the simplest possible RAG system, then identifies where it falls short.

In [ ]:
import chromadb
import requests
import numpy as np

client = chromadb.PersistentClient(path="./chroma_rag")

def get_embedding(text):
    resp = requests.post("http://localhost:11434/api/embeddings", json={
        "model": "nomic-embed-text",
        "prompt": text
    })
    return resp.json()["embedding"]

def ollama_generate(prompt):
    resp = requests.post("http://localhost:11434/api/generate", json={
        "model": "llama3.2",
        "prompt": prompt,
        "stream": False,
        "temperature": 0.1
    })
    return resp.json()["response"]

print("Ready")

## 1. Building Naive RAG

The simplest RAG: retrieve → concatenate → generate.

In [ ]:
# Create knowledge base
collection = client.get_or_create_collection(name="knowledge_base")

knowledge_docs = [
    {"id": "k1", "text": "LoRA (Low-Rank Adaptation) adds trainable rank decomposition matrices to attention layers."},
    {"id": "k2", "text": "QLoRA quantizes the base model to 4-bit and adds LoRA adapters on top."},
    {"id": "k3", "text": "RAG retrieves documents from a vector database and feeds them as context to an LLM."},
    {"id": "k4", "text": "Chunk overlapping ensures continuity between adjacent text chunks during retrieval."},
    {"id": "k5", "text": "HNSW is a graph-based ANN algorithm used by most vector databases."},
]

for doc in knowledge_docs:
    collection.add(
        ids=[doc["id"]],
        embeddings=[get_embedding(doc["text"])],
        documents=[doc["text"]]
    )

print(f"Knowledge base: {len(knowledge_docs)} docs")

## 2. The Naive RAG Function

In [ ]:
def naive_rag(query, top_k=2):
    # Step 1: Retrieve
    query_emb = get_embedding(query)
    results = collection.query(
        query_embeddings=[query_emb],
        n_results=top_k
    )
    
    # Step 2: Concatenate context
    context = "\n\n".join(results['documents'][0])
    
    # Step 3: Generate
    prompt = f"""Answer the question based on the context below.

Context:
{context}

Question: {query}
Answer:"""
    
    answer = ollama_generate(prompt)
    
    return answer, results['documents'][0]

# Test
answer, sources = naive_rag("What is LoRA?")
print(f"Answer: {answer}")
print(f"Sources: {sources}")

## 3. Problems with Naive RAG

| Problem | Effect |
|---|---|
| **Single query** | One query might miss the best documents |
| **Fixed chunk size** | Might split relevant info across chunks |
| **No reranking** | Top-2 might not be the most relevant |
| **Query-doc mismatch** | User query ≠ ideal search query |
| **Lost in the middle** | LLM ignores middle documents |

Let's demonstrate:

In [ ]:
# A query that naive RAG struggles with
answer, sources = naive_rag("How is QLoRA different from regular LoRA?")
print(f"Problem: Only retrieved docs about LoRA and RAG, but missing the QLoRA connection")
print(f"\nAnswer: {answer}")

## 4. What We'll Fix in Advanced RAG

| Technique | Fixes |
|---|---|
| HyDE (Hypothetical Document Embeddings) | Query-doc mismatch |
| Multi-Query | Single query misses relevant docs |
| Reranking | Bad ordering of retrieved docs |
| Sentence Window Retrieval | Chunk boundary issues |
| Self-RAG | Hallucination from irrelevant context |

**Next**: Advanced RAG techniques →